# Train pipeline

## Setup envoironment

### Colab

In [ ]:
!git clone https://github.com/trxxnk/text-image-alignment.git

In [ ]:
import os
os.chdir("/content/text-image-alignment/")
print(f"Working directory: {os.getcwd()}")

In [ ]:
!git checkout dev

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!chmod +x src/scripts/setup_colab.sh
!src/scripts/setup_colab.sh

### Local

In [1]:
import os, sys
from pathlib import Path

os.chdir(os.path.dirname(sys.prefix))
_repo = Path.cwd().resolve()
os.environ.setdefault("TORCH_HOME", str(_repo / ".cache" / "torch"))
print(f"Working directory: {os.getcwd()}")
print(f"TORCH_HOME={os.environ['TORCH_HOME']}")

Working directory: /home/trxxnk/mycode/diplom
TORCH_HOME=/home/trxxnk/mycode/diplom/.cache/torch


## Import libs

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.transforms import v2

import os
import json
import mlflow
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
from src.tps_dewarp.training import (
    Trainer,
    load_train_config,
    build_tps_dataloaders,
    build_model,
)

## Setup torch, dugshub, mlflow

In [4]:
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [5]:
import dagshub
dagshub.init(repo_owner='trxxnk', repo_name='text-image-alignment', mlflow=True)

Accessing as trxxnk

Initialized MLflow to track repo "trxxnk/text-image-alignment"

Repository trxxnk/text-image-alignment initialized!

## Dataset

### Load Dataset

In [6]:
CONFIG_PATH = "configs/train_smoke.yaml"
cfg = load_train_config(CONFIG_PATH)

In [8]:
train_loader, val_loader, test_loader = build_tps_dataloaders(cfg, device)
len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)

(26742, 3342, 3344)

In [9]:
# Проверка одного элемента train
img, delta, difficulty = train_loader.dataset[0]

print(img.shape)
print(delta.shape)
print(difficulty)


torch.Size([1, 256, 256])
torch.Size([81, 2])
identity


In [12]:
from collections import Counter

subset = train_loader.dataset
diffs = [subset.dataset.samples[i]["difficulty"] for i in subset.indices]
counter = Counter(diffs)
total = counter.total()
probas = [val / total for val in counter.values()]
print(f"""
  {counter}
  {total=}
  probas={[round(val, 2) for val in probas]}
"""
)


  Counter({'easy': 10809, 'medium': 5408, 'hard': 5320, 'identity': 5205})
  total=26742
  probas=[0.19, 0.2, 0.2, 0.4]



In [11]:
x, y, d = next(iter(train_loader))

print(x.shape)  # (B, 1, H, H)
print(y.shape)
print(len(d))

torch.Size([8, 1, 256, 256])
torch.Size([8, 81, 2])
8


## Model

### Load Model

In [13]:
model = build_model(cfg).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/trxxnk/mycode/diplom/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:06<00:00, 7.36MB/s]


In [14]:
x = train_loader.dataset[1][0]
x = x.unsqueeze(0).to(device)
out = model(x)
out.shape

torch.Size([1, 162])

In [15]:
train_loader.dataset[1][1].shape

torch.Size([81, 2])

## Train Loop

In [16]:
mlflow.set_experiment("TPS_Dewarp");
mlflow.start_run(run_name="04_test_local")

<ActiveRun: >

In [17]:
# Loss, optimizer, scheduler и цикл — внутри Trainer (см. configs/train_default.yaml)
trainer = Trainer(cfg, model, train_loader, val_loader, test_loader, device)

In [ ]:
# Resume: trainer.fit(resume_from="models/checkpoints/last.pt")
trainer.fit(resume_from=None)

In [ ]:
# Завершить эксперимент по run_id
import mlflow
run_id = "***"
client = mlflow.tracking.MlflowClient()
client.set_terminated(run_id)

🏃 View run 04_test_local at: https://dagshub.com/trxxnk/text-image-alignment.mlflow/#/experiments/0/runs/6c81b4c80d714aab8dd388f1af5b1f3c
🧪 View experiment at: https://dagshub.com/trxxnk/text-image-alignment.mlflow/#/experiments/0
